In [2]:
import csv
import pandas as pd
file_name = "L_Dataset.csv"
data = []
file = open(file_name, mode="r", encoding="latin1")
reader = csv.DictReader(file)
for row in reader:
    row["Category_Type"] = row.get("Category_Type", "").strip().title()
    row["Content_Format"] = row.get("Content_Format", "").strip().title()
    data.append(row)
file.close()
df = pd.DataFrame(data)
for col in ["Views", "Shares", "Saves", "Comments", "Likes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
master_triggers = [
    "crying",
    "soul",
    "hits",
    "omg",
    "😭",
    "felt",
    "hurt",
    "emotional",
    "profound",
    "chills",
    "trajectory",
    "painful",
    "intense",
    "suffering",
    "targeted",
    "me",
    "fr",
    "true",
    "literally",
    "said it",
    "needed",
    "validates",
    "seen",
    "clarity",
    "relatable",
    "perspective",
    "mirrors",
    "problem",
    "anxiety",
    "trap",
    "struggle",
    "cycle",
    "exposed",
    "wrong",
    "issue",
    "habits",
    "routines",
    "trapped",
]
comment_cols = [
    "User_Comment_1",
    "User_Comment_2",
    "User_Comment_3",
    "User_Comment_4",
    "User_Comment_5",
]


def get_comment_score(row):
    combined_text = " ".join(
        [
            str(row[c]).lower()
            for c in comment_cols
            if c in row and pd.notna(row[c])
        ]
    )
    return sum(1 for word in master_triggers if word in combined_text)
df["Problem_Awareness_Score"] = df.apply(get_comment_score, axis=1)
W_SHARE, W_SAVE, W_COMMENT, W_LIKE = 10.0, 10.0, 5.0, 1.0
df["Eng_Score"] = (
    (W_SHARE * df["Shares"])
    + (W_SAVE * df["Saves"])
    + (W_COMMENT * df["Comments"])
    + (W_LIKE * df["Likes"])
)
df["Post_Virality_Index"] = df["Eng_Score"] / (df["Views"] + 1)
grouped = (
    df.groupby(["Category_Type", "Content_Format"])
    .agg(
        Avg_Problem_Awareness=("Problem_Awareness_Score", "mean"),
        Avg_Virality_Index=("Post_Virality_Index", "mean"),
    )
    .reset_index()
)

pivoted = grouped.pivot(
    index="Category_Type", columns="Content_Format"
).round(2)
pivoted.columns = [f"{col[0]}_{col[1]}" for col in pivoted.columns]
summary_dashboard = pivoted.reset_index()

if "Avg_Problem_Awareness_Video" in summary_dashboard.columns:
    summary_dashboard = summary_dashboard.sort_values(
        by="Avg_Problem_Awareness_Video", ascending=False
    )
print("\nNLP AUDIENCE METRIC SUMMARY")
print(summary_dashboard.to_string(index=False))


NLP AUDIENCE METRIC SUMMARY
   Category_Type  Avg_Problem_Awareness_Short  Avg_Problem_Awareness_Video  Avg_Virality_Index_Short  Avg_Virality_Index_Video
         Fashion                         6.80                         9.00                      0.02                      0.07
   Mental Health                         6.60                         8.20                      0.08                      0.05
  Reaction Video                         6.43                         8.00                      0.05                      0.18
  Clothing Hauls                         6.33                         7.86                      0.03                      0.09
           Music                         7.20                         7.80                      0.02                      0.01
         Mukbang                         6.00                         7.75                      0.03                      0.07
   Dating Advice                         6.00                         7.50        